In [6]:
# ============================================================
# FAILURE CASE DEBUG — EPIC-Kitchens
# Run this ONLY to find failure cases
# Do NOT run figure code yet
# ============================================================

import numpy as np
import pandas as pd
import torch
import os

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# PATHS
# ============================================================
LABELS_CSV    = "../Labels/EPIC/P01_04_vgg_fused_labeled.csv"
AFTER_PROP    = "../Prediction/EPIC/P01_04_vgg_fused_after_propagation.csv"
BEFORE_PROP   = "../Prediction/EPIC/P01_04_vgg_fused_before_propagation.csv"
ANNOTATION    = r"D:\Datasets\Datasets\EPIC_Kitchen\Label\P01_04.csv"

# ============================================================
# LOAD
# ============================================================
labels_df    = pd.read_csv(LABELS_CSV)
y_true       = labels_df.drop(
                   columns=["frame_id"]
               ).values.astype(np.float32)
frame_ids    = labels_df["frame_id"].values
NUM_FRAMES, NUM_CLASSES = y_true.shape

after_df     = pd.read_csv(AFTER_PROP)
Y_star       = after_df.drop(
                   columns=["frame_id"]
               ).values.astype(np.float32)

before_df    = pd.read_csv(BEFORE_PROP)
Y_rw         = before_df.drop(
                   columns=["frame_id"]
               ).values.astype(np.float32)

ann_df       = pd.read_csv(ANNOTATION)
ann_map      = ann_df[[
    "ActionLabel", "ActionName"]
].drop_duplicates()
action_map   = dict(zip(
    ann_map["ActionLabel"].astype(int),
    ann_map["ActionName"]
))

# ============================================================
# CLASS FREQUENCIES
# ============================================================
class_freq = y_true.sum(axis=0)
print("="*60)
print("CLASS FREQUENCIES:")
print("="*60)
for cls in range(NUM_CLASSES):
    print(f"  Class {cls:2d} | "
          f"Freq: {class_freq[cls]:6.0f} | "
          f"Name: {action_map.get(cls, str(cls))}")

# ============================================================
# TOP 10 WORST FRAMES
# ============================================================
errors = np.abs(y_true - Y_star).sum(axis=1)
worst_frames = np.argsort(errors)[::-1][:10]

print("\n" + "="*60)
print("TOP 10 WORST PREDICTED FRAMES:")
print("="*60)
for t in worst_frames:
    gt_active  = np.where(y_true[t] == 1)[0]
    gt_names   = [action_map.get(
        int(c), str(c)) for c in gt_active]
    carrw_vals = Y_star[t, gt_active].round(3)
    rw_vals    = Y_rw[t, gt_active].round(3)
    print(f"\n  Frame {frame_ids[t]}:")
    print(f"  GT classes : {gt_names}")
    print(f"  CARRW scores: {carrw_vals}")
    print(f"  RW    scores: {rw_vals}")
    print(f"  Total error : {errors[t]:.4f}")

# ============================================================
# FALSE NEGATIVES — GT=1 but CARRW low
# ============================================================
print("\n" + "="*60)
print("FALSE NEGATIVES (GT=1, CARRW<0.4):")
print("="*60)
fn_list = []
for t in range(NUM_FRAMES):
    gt_active = np.where(y_true[t] == 1)[0]
    for cls in gt_active:
        if Y_star[t, cls] < 0.4:
            fn_list.append({
                "frame_idx":   t,
                "frame_id":    frame_ids[t],
                "cls":         int(cls),
                "cls_name":    action_map.get(
                    int(cls), str(cls)),
                "carrw_score": float(
                    Y_star[t, cls]),
                "rw_score":    float(
                    Y_rw[t, cls]),
                "gt_classes":  gt_active.tolist(),
                "cls_freq":    float(
                    class_freq[cls])
            })

print(f"Total false negatives: {len(fn_list)}")
for f in fn_list[:10]:
    print(f"\n  Frame {f['frame_id']}:")
    print(f"  Missed : {f['cls_name']}")
    print(f"  Freq   : {f['cls_freq']:.0f}")
    print(f"  CARRW  : {f['carrw_score']:.4f}")
    print(f"  RW     : {f['rw_score']:.4f}")
    print(f"  All GT : "
          f"{[action_map.get(int(c), str(c)) for c in f['gt_classes']]}")

# ============================================================
# FALSE POSITIVES — GT=0 but CARRW high
# ============================================================
print("\n" + "="*60)
print("FALSE POSITIVES (GT=0, CARRW>0.6):")
print("="*60)
fp_list = []
for t in range(NUM_FRAMES):
    for cls in range(NUM_CLASSES):
        if y_true[t, cls] == 0 and \
                Y_star[t, cls] > 0.6:
            fp_list.append({
                "frame_idx":   t,
                "frame_id":    frame_ids[t],
                "cls":         int(cls),
                "cls_name":    action_map.get(
                    int(cls), str(cls)),
                "carrw_score": float(
                    Y_star[t, cls]),
                "rw_score":    float(
                    Y_rw[t, cls]),
                "cls_freq":    float(
                    class_freq[cls])
            })

print(f"Total false positives: {len(fp_list)}")
for f in fp_list[:10]:
    print(f"\n  Frame {f['frame_id']}:")
    print(f"  Wrong  : {f['cls_name']}")
    print(f"  Freq   : {f['cls_freq']:.0f}")
    print(f"  CARRW  : {f['carrw_score']:.4f}")
    print(f"  RW     : {f['rw_score']:.4f}")

# ============================================================
# TRANSITION REGION FAILURES
# Find frames where action is starting or
# ending — model often fails here
# ============================================================
print("\n" + "="*60)
print("TRANSITION REGION FAILURES:")
print("="*60)
trans_list = []
for t in range(1, NUM_FRAMES-1):
    # Action starts at this frame
    started = np.where(
        (y_true[t] == 1) & 
        (y_true[t-1] == 0)
    )[0]
    # Action ends at this frame
    ended = np.where(
        (y_true[t] == 0) & 
        (y_true[t-1] == 1)
    )[0]

    for cls in started:
        if Y_star[t, cls] < 0.4:
            trans_list.append({
                "frame_idx":   t,
                "frame_id":    frame_ids[t],
                "cls":         int(cls),
                "cls_name":    action_map.get(
                    int(cls), str(cls)),
                "type":        "action_start",
                "carrw_score": float(
                    Y_star[t, cls]),
                "cls_freq":    float(
                    class_freq[cls])
            })

    for cls in ended:
        if Y_star[t, cls] > 0.6:
            trans_list.append({
                "frame_idx":   t,
                "frame_id":    frame_ids[t],
                "cls":         int(cls),
                "cls_name":    action_map.get(
                    int(cls), str(cls)),
                "type":        "action_end",
                "carrw_score": float(
                    Y_star[t, cls]),
                "cls_freq":    float(
                    class_freq[cls])
            })

print(f"Total transition failures: "
      f"{len(trans_list)}")
for f in trans_list[:10]:
    print(f"\n  Frame {f['frame_id']}:")
    print(f"  Class  : {f['cls_name']}")
    print(f"  Type   : {f['type']}")
    print(f"  Freq   : {f['cls_freq']:.0f}")
    print(f"  CARRW  : {f['carrw_score']:.4f}")

print("\n" + "="*60)
print("DEBUG COMPLETE")
print("="*60)

CLASS FREQUENCIES:
  Class  0 | Freq:    400 | Name: pull-down cup
  Class  1 | Freq:    135 | Name: put-down cup
  Class  2 | Freq:     38 | Name: take bag:cereal
  Class  3 | Freq:    181 | Name: fold bag:cereal
  Class  4 | Freq:    226 | Name: put bag:cereal
  Class  5 | Freq:    181 | Name: close box:cereal
  Class  6 | Freq:     76 | Name: open cupboard
  Class  7 | Freq:    183 | Name: put-down box:cereal
  Class  8 | Freq:     91 | Name: close cupboard
  Class  9 | Freq:    387 | Name: pull-down tablecloth
  Class 10 | Freq:    258 | Name: fold tablecloth
  Class 11 | Freq:     47 | Name: take-up liquid
  Class 12 | Freq:    139 | Name: pour-up liquid
  Class 13 | Freq:     93 | Name: open tap
  Class 14 | Freq:    653 | Name: rinse cup
  Class 15 | Freq:    220 | Name: rinse spoon
  Class 16 | Freq:     42 | Name: turn tap
  Class 17 | Freq:    160 | Name: rinse sink
  Class 18 | Freq:    151 | Name: rinse hand
  Class 19 | Freq:     41 | Name: close tap
  Class 20 | Freq:    

In [ ]:
# ============================================================
# FAILURE CASE FIGURE — EPIC-Kitchens
# Case 1: Frame 5892 — co-occurring actions
#         CARRW over-smooths drink water
# Case 2: Frame 407 — transition failure
#         CARRW keeps high confidence after
#         rare action ends
# ============================================================

import numpy as np
import pandas as pd
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from PIL import Image
import os
import base64
from io import BytesIO

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu")

# ============================================================
# PATHS
# ============================================================
LABELS_CSV    = "../Labels/EPIC/P01_04_vgg_fused_labeled.csv"
AFTER_PROP    = "../Prediction/EPIC/P01_04_vgg_fused_after_propagation.csv"
BEFORE_PROP   = "../Prediction/EPIC/P01_04_vgg_fused_before_propagation.csv"
ANNOTATION    = r"D:\Datasets\Datasets\EPIC_Kitchen\Label\P01_04.csv"
FRAMES_ROOT   = r"D:\Datasets\Datasets\EPIC_Kitchen\RGB\P01_04\Original"
SAVE_DIR      = "../Qualitative/EPIC/"
os.makedirs(SAVE_DIR, exist_ok=True)

# ============================================================
# LOAD
# ============================================================
labels_df    = pd.read_csv(LABELS_CSV)
y_true       = labels_df.drop(
                   columns=["frame_id"]
               ).values.astype(np.float32)
frame_ids    = labels_df["frame_id"].values
NUM_FRAMES, NUM_CLASSES = y_true.shape

after_df     = pd.read_csv(AFTER_PROP)
Y_star       = after_df.drop(
                   columns=["frame_id"]
               ).values.astype(np.float32)

before_df    = pd.read_csv(BEFORE_PROP)
Y_rw         = before_df.drop(
                   columns=["frame_id"]
               ).values.astype(np.float32)

ann_df       = pd.read_csv(ANNOTATION)
ann_map      = ann_df[[
    "ActionLabel", "ActionName"]
].drop_duplicates()
action_map   = dict(zip(
    ann_map["ActionLabel"].astype(int),
    ann_map["ActionName"]
))

def smooth(arr, window=5):
    kernel = np.ones(window) / window
    return np.convolve(arr, kernel, mode='same')

# ============================================================
# MANUALLY DEFINED FAILURE CASES
# ============================================================
CASE1_FRAME_ID = 5892
CASE2_FRAME_ID = 407

case1_idx = int(np.argmin(
    np.abs(frame_ids - CASE1_FRAME_ID)))
case2_idx = int(np.argmin(
    np.abs(frame_ids - CASE2_FRAME_ID)))

print(f"Case 1 frame idx: {case1_idx} "
      f"frame id: {frame_ids[case1_idx]}")
print(f"Case 2 frame idx: {case2_idx} "
      f"frame id: {frame_ids[case2_idx]}")

# Case 1 classes
CASE1_CLS_MISSED  = 25  # drink water
CASE1_CLS_CORRECT = 26  # put-down bottle

# Case 2 classes
CASE2_CLS = 2   # take bag:cereal

# Narrower window for Case 1
# to make failure more visible
WINDOW_CASE1 = 60
# Wider window for Case 2
WINDOW_CASE2 = 150

# ============================================================
# FIGURE
# ============================================================
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=[
        "Failure Case 1: Over-smoothing on "
        "Co-occurring Actions — "
        "CARRW misses \'drink water\' "
        "(Frame 5892)",
        "Failure Case 2: Temporal Boundary "
        "Failure — CARRW maintains high "
        "confidence after \'take bag:cereal\' "
        "ends (Frame 407)"
    ],
    vertical_spacing=0.18
)

# ============================================================
# CASE 1 PLOT
# ============================================================
f_start1 = max(0, case1_idx - WINDOW_CASE1//2)
f_end1   = min(NUM_FRAMES,
               case1_idx + WINDOW_CASE1//2)
frames1  = frame_ids[f_start1:f_end1]

# GT — drink water
fig.add_trace(
    go.Scatter(
        x=frames1,
        y=y_true[f_start1:f_end1,
                 CASE1_CLS_MISSED],
        mode="lines",
        line=dict(
            color="black",
            width=1.5,
            dash="dot"),
        name="GT: drink water",
        legendgroup="case1",
        showlegend=True
    ),
    row=1, col=1
)

# RW — drink water (correct)
fig.add_trace(
    go.Scatter(
        x=frames1,
        y=smooth(Y_rw[f_start1:f_end1,
                      CASE1_CLS_MISSED]),
        mode="lines",
        line=dict(
            color="#1565C0",
            width=2.0,
            dash="dash"),
        name="RW: drink water (correct)",
        legendgroup="case1",
        showlegend=True
    ),
    row=1, col=1
)

# CARRW — drink water (missed)
fig.add_trace(
    go.Scatter(
        x=frames1,
        y=smooth(Y_star[f_start1:f_end1,
                        CASE1_CLS_MISSED]),
        mode="lines",
        line=dict(
            color="#E53935",
            width=2.5),
        name="CARRW: drink water (missed)",
        legendgroup="case1",
        showlegend=True
    ),
    row=1, col=1
)

# CARRW — put-down bottle (correct)
fig.add_trace(
    go.Scatter(
        x=frames1,
        y=smooth(Y_star[f_start1:f_end1,
                        CASE1_CLS_CORRECT]),
        mode="lines",
        line=dict(
            color="#43A047",
            width=2.0),
        name="CARRW: put-down bottle "
             "(correct)",
        legendgroup="case1",
        showlegend=True
    ),
    row=1, col=1
)

# Detection threshold line
fig.add_hline(
    y=0.5,
    line=dict(
        color="gray",
        width=1.0,
        dash="dot"),
    annotation_text="threshold = 0.5",
    annotation_font=dict(
        size=9,
        family="Times New Roman",
        color="gray"
    ),
    row=1, col=1
)

# Failure point marker
fig.add_vline(
    x=float(frame_ids[case1_idx]),
    line=dict(
        color="#E53935",
        width=1.5,
        dash="dash"),
    row=1, col=1
)

# Failure region shading
fig.add_vrect(
    x0=float(frame_ids[
        max(0, case1_idx-10)]),
    x1=float(frame_ids[
        min(NUM_FRAMES-1, case1_idx+10)]),
    fillcolor="#E53935",
    opacity=0.10,
    layer="below",
    line_width=0,
    row=1, col=1
)

# Annotation
fig.add_annotation(
    x=float(frame_ids[case1_idx]),
    y=1.12,
    xref="x", yref="y",
    text="<b>Over-smoothing</b>",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#E53935",
    font=dict(
        color="#E53935",
        size=10,
        family="Times New Roman"
    )
)

# ============================================================
# CASE 2 PLOT
# ============================================================
f_start2 = max(0, case2_idx - WINDOW_CASE2//2)
f_end2   = min(NUM_FRAMES,
               case2_idx + WINDOW_CASE2//2)
frames2  = frame_ids[f_start2:f_end2]

# GT — take bag:cereal
fig.add_trace(
    go.Scatter(
        x=frames2,
        y=y_true[f_start2:f_end2,
                 CASE2_CLS],
        mode="lines",
        line=dict(
            color="black",
            width=1.5,
            dash="dot"),
        name="GT: take bag:cereal",
        legendgroup="case2",
        showlegend=True
    ),
    row=2, col=1
)

# RW — take bag:cereal
fig.add_trace(
    go.Scatter(
        x=frames2,
        y=smooth(Y_rw[f_start2:f_end2,
                      CASE2_CLS]),
        mode="lines",
        line=dict(
            color="#1565C0",
            width=2.0,
            dash="dash"),
        name="RW: take bag:cereal",
        legendgroup="case2",
        showlegend=True
    ),
    row=2, col=1
)

# CARRW — take bag:cereal (delayed)
fig.add_trace(
    go.Scatter(
        x=frames2,
        y=smooth(Y_star[f_start2:f_end2,
                        CASE2_CLS]),
        mode="lines",
        line=dict(
            color="#8E24AA",
            width=2.5),
        name="CARRW: take bag:cereal "
             "(delayed response)",
        legendgroup="case2",
        showlegend=True
    ),
    row=2, col=1
)

# Detection threshold line
fig.add_hline(
    y=0.5,
    line=dict(
        color="gray",
        width=1.0,
        dash="dot"),
    annotation_text="threshold = 0.5",
    annotation_font=dict(
        size=9,
        family="Times New Roman",
        color="gray"
    ),
    row=2, col=1
)

# Failure point marker
fig.add_vline(
    x=float(frame_ids[case2_idx]),
    line=dict(
        color="#8E24AA",
        width=1.5,
        dash="dash"),
    row=2, col=1
)

# Failure region shading
fig.add_vrect(
    x0=float(frame_ids[
        max(0, case2_idx-10)]),
    x1=float(frame_ids[
        min(NUM_FRAMES-1, case2_idx+10)]),
    fillcolor="#8E24AA",
    opacity=0.10,
    layer="below",
    line_width=0,
    row=2, col=1
)

# Annotation
fig.add_annotation(
    x=float(frame_ids[case2_idx]),
    y=1.12,
    xref="x2", yref="y2",
    text="<b>Delayed response</b>",
    showarrow=True,
    arrowhead=2,
    arrowcolor="#8E24AA",
    font=dict(
        color="#8E24AA",
        size=10,
        family="Times New Roman"
    )
)

# Action end annotation
fig.add_annotation(
    x=float(frame_ids[case2_idx]),
    y=-0.08,
    xref="x2", yref="y2",
    text="action ends here",
    showarrow=False,
    font=dict(
        color="gray",
        size=9,
        family="Times New Roman"
    )
)

# ============================================================
# UPDATE AXES
# ============================================================
for row in [1, 2]:
    fig.update_yaxes(
        range=[-0.15, 1.25],
        title_text="Confidence",
        title_font=dict(
            size=12,
            family="Times New Roman"
        ),
        tickfont=dict(
            size=11,
            family="Times New Roman"
        ),
        showgrid=True,
        gridcolor="rgba(0,0,0,0.08)",
        zeroline=True,
        zerolinecolor="rgba(0,0,0,0.2)",
        zerolinewidth=0.8,
        showline=True,
        linecolor="black",
        linewidth=0.8,
        row=row, col=1
    )
    fig.update_xaxes(
        title_text="Frame Index",
        title_font=dict(
            size=12,
            family="Times New Roman"
        ),
        tickfont=dict(
            size=11,
            family="Times New Roman"
        ),
        showgrid=False,
        showline=True,
        linecolor="black",
        linewidth=0.8,
        row=row, col=1
    )

# ============================================================
# LAYOUT
# ============================================================
fig.update_layout(
    plot_bgcolor="white",
    paper_bgcolor="white",
    font=dict(
        family="Times New Roman",
        size=11
    ),
    height=750,
    width=1000,
    title=dict(
        text="Failure Case Analysis — "
             "EPIC-Kitchens (P01\\_04)",
        font=dict(
            size=13,
            family="Times New Roman"
        ),
        x=0.5
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=-0.18,
        xanchor="center",
        x=0.5,
        font=dict(size=10),
        bgcolor="rgba(0,0,0,0)",
        borderwidth=0
    ),
    margin=dict(t=70, b=120, l=80, r=40)
)

# Subplot title font
for ann in fig.layout.annotations:
    ann.font = dict(
        size=10,
        family="Times New Roman",
        color="black"
    )

# ============================================================
# SAVE
# ============================================================
# fig.write_image(
#     f"{SAVE_DIR}P01_04_failure_cases.png",
#     scale=3
# )
fig.write_image(
    f"{SAVE_DIR}P01_04_failure_cases.eps",
    scale=3
)
fig.show()
print("Saved.")

Case 1 frame idx: 5892 frame id: 5892
Case 2 frame idx: 407 frame id: 407
